## rag(retrival augmented generation) from scratch

### ingestion pipeline

In [ ]:
# loading text/data from a file(pdf)
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader("../data/", glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = dir_loader.load()
print(documents)

In [ ]:
# creating chunks from text loaded

from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_and_chunk(documents, chunk_size=1000, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    return split_docs


chunks = process_and_chunk(documents)
# chunks

In [ ]:
# embedding
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, PreTrainedModel
import numpy as np
from typing import List, Any, Dict


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding manager: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"model loaded with embedding dimentions: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded!")
        print(f"generating embedding for {len(texts)} chunks..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

In [ ]:
# vector store
import os
import chromadb
import hashlib
import numpy as np
from pathlib import Path

class VectorManager:
    def __init__(self, collection_name=None, persist_directory="../data/vector_store"):
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        source_files = sorted({Path(doc.metadata["source"]).resolve() for doc in documents})
        digest = hashlib.sha256(embedding_manager.model_name.encode("utf-8"))
        for source_file in source_files:
            digest.update(str(source_file).encode("utf-8"))
            stat = source_file.stat()
            digest.update(f"{stat.st_size}:{stat.st_mtime}".encode("utf-8"))
        self.collection_name = collection_name or f"motor_docs_{digest.hexdigest()[:12]}"
        self._load_store()

    def _load_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "electric motor reference PDFs", "hnsw:space": "cosine"},
            )
            print(f"vector store initialized, collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error loading vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("no of documents does not match no of embedding")
        ids, metadatas, documents_text, embeddings_list = [], [], [], []
        for doc, embedding in zip(documents, embeddings):
            source = str(doc.metadata.get("source", "unknown"))
            page = doc.metadata.get("page", 0)
            doc_hash = hashlib.sha256(doc.page_content.encode("utf-8")).hexdigest()[:16]
            doc_id = hashlib.sha256(f"{source}:{page}:{doc_hash}".encode("utf-8")).hexdigest()[:24]
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        self.collection.upsert(ids=ids, metadatas=metadatas, documents=documents_text, embeddings=embeddings_list)
        print(f"added {len(documents)} documents into vector store")
        print(f"total documents in collection: {self.collection.count()}")

vector_store=VectorManager()
vector_store


In [ ]:
# convert text into embeddings
texts = [doc.page_content for doc in chunks]

# generate embedding
embeddings = embedding_manager.generate_embedding(texts)

# store in vector db
vector_store.add_documents(chunks, embeddings)

### retrieval pipeline

In [ ]:
class RagRetriever:
    def __init__(self, vector_store: VectorManager, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.30) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, self.vector_store.collection.count()),
            include=["documents", "metadatas", "distances"],
        )
        retrieved_docs = []
        documents = results["documents"][0] or []
        metadatas = results["metadatas"][0] or []
        distances = results["distances"][0] or []
        ids = results["ids"][0] or []
        for rank, (doc_id, document, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances), start=1
        ):
            similarity_score = 1 - float(distance)
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": doc_id, "content": document, "metadata": metadata,
                    "similarity_score": similarity_score, "rank": rank,
                })
        return retrieved_docs

rag_retriever = RagRetriever(vector_store, embedding_manager)
rag_retriever


### vector store to llm output generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GroqLLM:
    def __init__(self, model_name: str = os.getenv("GROQ_MODEL", "llama3-8b-8192"), api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a strict QA assistant. Answer the user's question using only the provided context. If the answer cannot be found in the context, reply exactly with: 'I do not have access to this information.' Do not use any outside knowledge.

Context:
{context}

Question: {question}
"""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
        

In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    llm = None

In [ ]:
rag_retriever.retrieve("what are synchronous machines")

In [ ]:
# llm integration
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import os
import re
import time
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API") or os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name=os.getenv("GROQ_MODEL", "llama3-8b-8192"), temperature=0.1, max_tokens=1024)
NO_ANSWER = "I couldn't find that in the electric motor reference material."
SYSTEM_PROMPT = """You are an intelligent AI assistant powered by a
Retrieval-Augmented Generation (RAG) system.

YOUR MAIN RESPONSIBILITY:
Understand the user's actual intention and provide
accurate, helpful, natural, and easy-to-understand answers.

1. UNDERSTANDING USER QUESTIONS
- Understand simple, casual and informal language.
- Do not require users to use technical terminology.
- Understand spelling mistakes and grammatical errors.
- Understand incomplete and poorly structured questions.
- Identify the actual meaning behind the question.
- Understand synonyms and alternative expressions.
- Do not depend on exact keyword matching.
- Interpret what the user means, not just what they type.

2. QUERY INTERPRETATION
- Convert unclear questions into meaningful search queries.
- Identify important concepts and related terminology.
- Consider different possible meanings of a question.
- Search for relevant information even when the user's
  wording differs from the document's wording.
- Never change the original intent of the question.

3. CONTEXT AWARENESS
- Remember previous questions in the conversation.
- Understand follow-up questions such as:
  "Explain more", "Why?", "How does it work?",
  "Give an example", and "What about its types?"
- Connect follow-up questions to the previous topic.
- Do not ask users to repeat information unnecessarily.

4. DOCUMENT RETRIEVAL
- Use the retrieved document context carefully.
- Consider all relevant retrieved passages.
- Combine information from multiple relevant passages.
- Do not select information based only on keyword matches.
- Ignore irrelevant retrieved content.
- If information is missing, clearly acknowledge it.

5. ANSWER GENERATION
- Answer the exact question asked.
- Use simple and understandable English.
- Explain technical terms in ordinary language.
- Prefer clear and direct answers.
- Break complicated concepts into smaller parts.
- Provide examples wherever useful.
- Use bullet points and headings when appropriate.
- Avoid unnecessary technical jargon.
- Do not give unnecessarily long answers.

6. ACCURACY AND HONESTY
- Never invent facts, figures, or references.
- Do not assume information that is not provided.
- Distinguish document-based facts from general knowledge.
- If the retrieved context does not contain the answer,
  clearly say that the documents do not provide it.
- Never pretend to have found information that is missing.

7. CONVERSATIONAL BEHAVIOUR
- Be friendly, natural and helpful.
- Respond like a human tutor.
- Do not sound robotic.
- Handle greetings and casual conversation naturally.
- Ask a clarification question only when necessary.
- If the question is understandable, answer directly.

8. EXPLANATION STYLE
When explaining a topic:
- Start with a simple definition.
- Explain how it works.
- Give a practical example.
- Explain important types or applications if relevant.
- Provide a short summary when useful.

9. OUTPUT QUALITY
- Prioritize relevance over answer length.
- Do not repeat the same information.
- Use examples related to the question.
- Make answers understandable even to beginners.

IMPORTANT:
Your primary purpose is to help users understand
information, not simply repeat retrieved document text."""

def format_bare_math(text):
    text = re.sub(
        r"\\\[(.*?)\\\]",
        lambda match: "$$\n" + match.group(1).strip() + "\n$$",
        text,
        flags=re.DOTALL,
    )
    text = re.sub(
        r"\\\((.*?)\\\)",
        lambda match: "$" + match.group(1).strip() + "$",
        text,
        flags=re.DOTALL,
    )

    def wrap_bare_formula(match):
        formula = match.group(1).strip().rstrip(" ;,.")
        return f"${formula}$"

    return re.sub(
        r"(?<!\\)\[\s*([^\[\]\n]*?(?:\\[A-Za-z]+|=)[^\[\]\n]*?)\s*(?<!\\)\]",
        wrap_bare_formula,
        text,
    )

def rag_simple(query, retriever, llm, top_k=5, min_score=0.30):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return NO_ANSWER

    context = "\n\n".join(
        f"[{index}] {doc['content']}" for index, doc in enumerate(results, start=1)
    )
    prompt = f"""Use the following retrieved passages to answer the question.
If no passage contains the answer, reply exactly: {NO_ANSWER}
Include the supporting passage number, such as [1].
Write equations in LaTeX using $...$ inline or $$...$$ on separate lines; do not use bare square brackets as math delimiters.

Passages:
{context}

Question: {query}
Answer:"""
    for attempt in range(3):
        try:
            response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=prompt)])
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    answer = response.content
    if not isinstance(answer, str):
        answer = "".join(str(part) for part in answer)
    answer = answer.strip()
    return format_bare_math(answer)


In [ ]:
answer = rag_simple("what is a synchronous motor", rag_retriever, llm)
from IPython.display import Markdown, display
display(Markdown(answer))

In [ ]:
# advance features and enhancements

# def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    
